In [3]:
import Bio
import Bio.motifs as motifs
from Bio import SeqIO
import numpy as np
import xgboost as xgb
import argparse, sys,time,os
import multiprocessing
import pandas as pd

In [13]:
# Parse the motif
with open('../data/cisBP_mouse.homer') as f: 
    parsed_motifs = motifs.parse(f, 'pfm-four-columns')

        0      1      2      3      4      5      6      7
G:  -4.70  -4.70  -4.70  -2.05  -1.17   1.18   0.90  -0.63
A:  -4.70   1.88   1.96  -2.84  -2.05   0.64  -1.17  -2.05
T:   1.92  -2.05  -4.70   1.80   1.71  -2.84  -1.17   0.81
C:  -2.84  -4.70  -4.70  -2.84  -4.70  -4.70   0.33   0.44

        0      1      2      3
G:   0.01   0.11  -0.08  -1.27
A:   0.15   0.66  -0.39   0.19
T:  -0.07  -0.60  -1.40  -4.04
C:  -0.10  -0.56   0.94   1.25

        0      1      2      3      4      5      6
G:  -1.39  -0.43  -1.67  -0.86  -0.23   0.82   0.14
A:  -0.28   1.31   1.64  -1.30  -0.60   0.31  -0.31
T:   1.22  -1.28  -0.99   1.35   0.83  -0.84  -0.18
C:  -1.12  -1.47  -3.96  -1.02  -0.50  -1.20   0.28

        0      1      2      3      4      5      6      7      8      9
G:   0.11  -0.88  -3.75  -2.40  -4.18  -3.04   0.25  -0.20   0.36  -0.25
A:   0.15  -0.88  -1.57   1.85   1.92  -2.57  -1.64   1.49  -0.53  -0.59
T:  -0.76   0.38   1.70  -3.24  -3.10   1.84   1.19  -2.20  -0.72   

In [23]:
m = parsed_motifs[1]
pssm = m.counts.normalize(pseudocounts=0.01).log_odds()
print(pssm)

log_odds = np.array(
            [ [pssm[letter][i] for letter in "ACGT"] for i in range(len(m)) ], # len(m) is the number of posistion
            dtype=np.float64
        )
print(log_odds)

        0      1      2      3
G:   0.01   0.11  -0.08  -1.27
A:   0.15   0.66  -0.39   0.19
T:  -0.07  -0.60  -1.40  -4.04
C:  -0.10  -0.56   0.94   1.25

[[ 0.14944424 -0.10040804  0.01042667 -0.07273256]
 [ 0.66028903 -0.56057699  0.11428845 -0.60136767]
 [-0.38792899  0.93740269 -0.08494636 -1.40401081]
 [ 0.19494872  1.25116365 -1.27184592 -4.03851394]]


In [27]:
multiprocessing.cpu_count()

64

## Test Run everything before XGB

In [59]:
######## METHODS ########

#### 0. Set up global var for multiprocessing workers (Pool) <- will be passed by fun1 ####
MOTIFS_LOG_ODDS = None
MOTIFS_THRESHOLDS = None
MOTIFS_LENGTHS = None

def init_worker(log_odds_list, thresholds, lengths):
    """Initialize worker process with motif data."""
    global MOTIFS_LOG_ODDS, MOTIFS_THRESHOLDS, MOTIFS_LENGTHS
    MOTIFS_LOG_ODDS = log_odds_list
    MOTIFS_THRESHOLDS = thresholds
    MOTIFS_LENGTHS = lengths
    

#### 1. parse the motif file ####
"""
motif file (PFM): 4 columns - ACGT
- we want to parse for pfm for each TF (using the threshold and ACGT weight)
- then we want to convert it to PWM -> PSSM/logodds matrix

return: list of PSSM, list of threshold, list of length of motif
"""
def parse_motifs(path):

    with open(path) as f:
        parsed_motifs = motifs.parse(f, fmt='pfm-four-columns')
    # print(f'Parsed {len(parsed_motifs)} motifs...'

    log_odds_list = []
    thresholds = []
    lengths = []
    
    for m in parsed_motifs: # m is one motif
        # Extract threshold from name (will be the 3rd field)
        name_parts = m.name.split('\t')
        threshold = float(name_parts[-1])

        # m.counts = PFM
        # PFM -> PWM -> PSSM (logodds)
        pssm = m.counts.normalize(pseudocounts=0.01).log_odds()
        
        # Convert to numpy array format for Bio.motifs._pwm.calculate (same as what is indicated in "https://github.com/biopython/biopython/blob/master/Bio/motifs/matrix.py#L402")
        # Shape: (length, 4) with columns A, C, G, T
        log_odds = np.array(
            [ [pssm[letter][i] for letter in "ACGT"] for i in range(len(m)) ], # len(m) is the number of posistion
            dtype=np.float64
        )
        
        log_odds_list.append(log_odds)
        thresholds.append(threshold)
        lengths.append(len(m))
    
    return log_odds_list, np.array(thresholds, dtype=np.float32), np.array(lengths, dtype=np.int32)


#### 2. Extract sequences from genome (bed format) ####
def extract_sequences(bed, genome_fa):
    sequences = []
    for idx, row in bed.iterrows():
        chrom = row['chrom']
        start_pos = row['start']
        end_pos = row['end']
        try:
            seq = str(genome_fa[chrom][start_pos:end_pos]).upper()
            sequences.append(seq)
        except:
            sequences.append('')
    return sequences


#### 3. Featurize sequence ####
def featurize_single_sequence(seq):
    """
    Featurize single sequence -> 1. binary hit and 2. max_score per motif within the sequence (effect size)

    Return: per motif -> length of 2 vector; across the motifs
    """

    # Get global var in parallel
    global MOTIFS_LOG_ODDS, MOTIFS_THRESHOLDS, MOTIFS_LENGTHS
    num_motifs = len(MOTIFS_LOG_ODDS)
    
    if not seq: #empty
        return np.zeros(num_motifs * 2, dtype=np.float32) #2 features per motif

    
    seq_bytes = bytes(seq, 'ASCII') # specify ASCII encoding
    seq_len = len(seq_bytes)

    # Make reverse complement for the sequence --> so that it will match the __sequence for TF binding motif__
    rc_seq_bytes = seq_bytes.translate(
        bytes.maketrans(b'ACGTacgtNn', b'TGCAtgcaNn')
    )[::-1]
    
    features = np.zeros(num_motifs * 2, dtype=np.float32)

    # Across all the motifs for this sequence
    for m_idx in range(num_motifs):
        log_odds = MOTIFS_LOG_ODDS[m_idx]
        threshold = MOTIFS_THRESHOLDS[m_idx]
        m_len = MOTIFS_LENGTHS[m_idx]
        
        if seq_len < m_len: #only process sequence longer than the motif
            continue
        
        n_positions = seq_len - m_len + 1

        ## [SLIDE OVER BOTH STRAND - forward and reverse]
        scores_fwd = np.empty(n_positions, np.float32)
        motifs._pwm.calculate(seq_bytes, log_odds, scores_fwd) # use the PSSM to slide, store result to the np array
        scores_rc = np.empty(n_positions, np.float32)
        motifs._pwm.calculate(rc_seq_bytes, log_odds, scores_rc)

        # transformation
        max_score = max(np.max(scores_fwd), np.max(scores_rc))
        has_hit = 1.0 if max_score >= threshold else 0.0
        
        # store
        base_idx = m_idx * 2
        features[base_idx] = has_hit
        features[base_idx + 1] = max_score
    
    return features


#### 4. Multiprocessing setup ####
def featurize_batch_worker(batch):
    """Worker function to featurize a batch of sequences."""
    start_idx, sequences = batch
    
    num_motifs = len(MOTIFS_LOG_ODDS)
    n_seqs = len(sequences)
    features = np.zeros((n_seqs, num_motifs * 2), dtype=np.float32)
    
    for i, seq in enumerate(sequences):
        features[i] = featurize_single_sequence(seq)
    
    return (start_idx, features)


def featurize_sequences_parallel(sequences, log_odds_list, thresholds, lengths, n_workers=None):
    """Featurize all sequences in parallel using multiprocessing.Pool with initializer."""
    if n_workers is None:
        n_workers = multiprocessing.cpu_count()
    
    n_samples = len(sequences)
    num_motifs = len(log_odds_list)
    n_features = num_motifs * 2
    X = np.zeros((n_samples, n_features), dtype=np.float32)
    
    # Create batches - more batches for better load balancing
    batch_size = max(1, n_samples // (n_workers * 4)) #allow workers picking up lower jobs (4x as sweet spot)
    batches = []
    for i in range(0, n_samples, batch_size):
        end_idx = min(i + batch_size, n_samples)
        batches.append((i, sequences[i:end_idx]))
    
    # Use Pool with initializer to pass motif data to workers
    with multiprocessing.Pool(n_workers, initializer=init_worker, initargs=(log_odds_list, thresholds, lengths)) as pool:
        results = pool.map(featurize_batch_worker, batches)
    
    for start_idx, features in results:
        end_idx = start_idx + features.shape[0]
        X[start_idx:end_idx] = features
    
    return X

In [60]:
args = argparse.Namespace(
    genome = '../data/mm10.fa',
    train = '../data/train.B.bed',
    motifs = '../data/cisBP_mouse.homer',
    predict = '../data/test.B.bed',
    output_file = 'test.csv'
)
print(args.genome, args.train, args.output_file)

../data/mm10.fa ../data/train.B.bed test.csv


In [61]:
start_time = time.time()
    
### Load data ####
# Load motifs
print(f"Loading motifs from {args.motifs}...")
log_odds_list, thresholds, lengths = parse_motifs(args.motifs)
print(f">> Loaded {len(log_odds_list)} motifs")

# Load genome
print(f"Loading genome from {args.genome}...")
genome_fa = {}
for seqr in SeqIO.parse(args.genome, 'fasta'):
    genome_fa[seqr.name] = seqr.seq    
print(f">> Loaded {len(genome_fa.keys())} chromosomes")

# Load training data
print(f"Loading training data from {args.train}...")
train_df = pd.read_csv(args.train, sep='\t', header=None, 
                       names=['chrom', 'start', 'end', 'label'])
train_df = train_df.reset_index(drop=True)
y_train = train_df['label'].values
print(f">> Loaded {len(train_df)} training samples")

# Load test data
print(f"Loading test data from {args.predict}...")
test_df = pd.read_csv(args.predict, sep='\t', header=None,
                      names=['chrom', 'start', 'end'])
test_df = test_df.reset_index(drop=True)
print(f">> Loaded {len(test_df)} test samples")

# Extract sequences
print("Extracting training sequences...")
train_sequences = extract_sequences(train_df, genome_fa)
print("Extracting test sequences...")
test_sequences = extract_sequences(test_df, genome_fa)


#### Featurizing ####
# Featurize training data in parallel
print("Featurizing training data...")
feat_start = time.time()
X_train = featurize_sequences_parallel(train_sequences, log_odds_list, thresholds, lengths)
print(f">> Featurization took {time.time() - feat_start:.2f}s")
print(f">> Feature matrix shape: {X_train.shape}")

# Featurize test data in parallel
print("Featurizing test data...")
feat_start = time.time()
X_test = featurize_sequences_parallel(test_sequences, log_odds_list, thresholds, lengths)
print(f">> Featurization took {time.time() - feat_start:.2f}s")

Loading motifs from ../data/cisBP_mouse.homer...
>> Loaded 862 motifs
Loading genome from ../data/mm10.fa...
>> Loaded 66 chromosomes
Loading training data from ../data/train.B.bed...
>> Loaded 461061 training samples
Loading test data from ../data/test.B.bed...
>> Loaded 51534 test samples
Extracting training sequences...
Extracting test sequences...
Featurizing training data...
>> Featurization took 2372.17s
>> Feature matrix shape: (461061, 1724)
Featurizing test data...
>> Featurization took 279.32s


In [62]:
train_df, genome_fa

(       chrom     start       end     label
 0       chr1   3020661   3020911  0.341588
 1       chr1   3087101   3087351 -0.230986
 2       chr1   3119984   3120234 -0.006822
 3       chr1   3121360   3121610 -0.172422
 4       chr1   3372662   3372912 -0.266138
 ...      ...       ...       ...       ...
 461056  chrY  90812325  90812575  2.116761
 461057  chrY  90812781  90813031  2.077713
 461058  chrY  90813050  90813300  0.386363
 461059  chrY  90813499  90813749  0.057661
 461060  chrY  90828860  90829110  0.468798
 
 [461061 rows x 4 columns],
 {'chr1': Seq('NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...NNN'),
  'chr10': Seq('NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...NNN'),
  'chr11': Seq('NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...NNN'),
  'chr12': Seq('NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...NNN'),
  'chr13': Seq('NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...NNN'),
  'chr14': Seq('NNNNNNNNNNNNNNNNNNNNNNNNNNN

In [63]:
X_train, X_test.shape

(array([[0.        , 0.90059793, 1.        , ..., 4.203666  , 1.        ,
         5.2300506 ],
        [1.        , 6.156305  , 1.        , ..., 5.241885  , 1.        ,
         5.3600693 ],
        [1.        , 6.0709467 , 1.        , ..., 4.9160314 , 1.        ,
         7.7405562 ],
        ...,
        [0.        , 3.2702043 , 1.        , ..., 4.2008133 , 1.        ,
         3.8036146 ],
        [0.        , 2.5831926 , 1.        , ..., 4.6645923 , 1.        ,
         5.1070027 ],
        [0.        , 1.4818693 , 1.        , ..., 3.6426752 , 1.        ,
         5.2300506 ]], dtype=float32),
 (51534, 1724))

In [64]:
zero_cols = (X_train.sum(axis=0) == 0).sum()
print(f"Zero-variance features: {zero_cols} / {X_train.shape[1]}")

# How many motifs had any hits at all?
hit_cols = X_train[:, ::2]  # binary hit features (every other column)
print(f"Motifs with at least one hit: {(hit_cols.sum(axis=0) > 0).sum()} / {hit_cols.shape[1]}")

Zero-variance features: 0 / 1724
Motifs with at least one hit: 862 / 862


## XGB grid search

In [66]:
y_train

array([ 0.341588, -0.230986, -0.006822, ...,  0.386363,  0.057661,
        0.468798])

In [67]:
print(pd.Series(y_train).describe())
print(f"Variance: {np.var(y_train):.6f}")

count    461061.000000
mean          0.707627
std           0.940233
min          -2.195210
25%           0.090905
50%           0.477699
75%           1.065227
max           6.461746
dtype: float64
Variance: 0.884035


In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import make_scorer
from scipy.stats import pearsonr

# Custom scorer
def pearson_r(y_true, y_pred):
    r, _ = pearsonr(y_true, y_pred)
    return r
scorer = make_scorer(pearson_r)

# Setup GridSearchCV
kf = KFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7], #default = 3
    'learning_rate': [0.1, 0.01],
    'subsample': [0.5, 0.8, 1] #SGD when not 1
}
xgb_reg = xgb.XGBRegressor(
    random_state=42, n_jobs=-1, #scoring in grid search
    tree_method='hist',
)

grid_search = GridSearchCV(
    xgb_reg,
    param_grid,
    cv=kf,
    scoring=scorer,
    n_jobs=-1,
    refit=True,
    verbose=3,
)

# Fit
grid_search.fit(X_train, y_train)

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best Pearson r:{grid_search.best_score_:.4f}")
